# Experiment: Yeo-Johnson Transformation & Robust Loss Regressor

This experiment tests target standardization using power transforms alongside models optimized on Huber robust loss functions to stabilize outliers.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_predict
from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

train_path = '/content/train.csv'
if not os.path.exists(train_path):
    raise FileNotFoundError("Please upload train.csv to /content/ first!")

train_df = pd.read_csv(train_path)
y_dev_full = train_df['SalePrice'].copy()
X_dev_full = train_df.drop(columns=['SalePrice', 'Id'])

X_dev_local, _, y_dev_local, _ = train_test_split(
    X_dev_full, y_dev_full, test_size=0.20, random_state=42
)
cv_local = KFold(n_splits=5, shuffle=True, random_state=42)

class HouseFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X_copy = X.copy()
        X_copy['TotalSF'] = X_copy['TotalBsmtSF'].fillna(0) + X_copy['1stFlrSF'].fillna(0) + X_copy['2ndFlrSF'].fillna(0)
        X_copy['TotalBath'] = (
            X_copy['FullBath'].fillna(0) +
            (0.5 * X_copy['HalfBath'].fillna(0)) +
            X_copy['BsmtFullBath'].fillna(0) +
            (0.5 * X_copy['BsmtHalfBath'].fillna(0))
        )
        X_copy['HouseAge'] = X_copy['YrSold'] - X_copy['YearBuilt']
        return X_copy

X_dev_engineered = HouseFeatureEngineer().transform(X_dev_local)
numerical_cols = [c for c in X_dev_engineered.columns if X_dev_engineered[c].dtype in ['int64', 'float64'] and c != 'MSSubClass']
categorical_cols = [c for c in X_dev_engineered.columns if X_dev_engineered[c].dtype == 'object'] + ['MSSubClass']

na_meaningful_cols = [
    'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType'
]
actual_na_cols = [c for c in na_meaningful_cols if c in categorical_cols]
actual_standard_cat_cols = [c for c in categorical_cols if c not in na_meaningful_cols]

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
    ('scaler', RobustScaler())
])

categorical_transformer_standard = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

categorical_transformer_na_valid = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

local_preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_cols),
    ('cat_std', categorical_transformer_standard, actual_standard_cat_cols),
    ('cat_na', categorical_transformer_na_valid, actual_na_cols)
])


In [2]:
from sklearn.preprocessing import PowerTransformer
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor

# Configure Gradient Boosting optimizing for Huber Loss inside a Yeo-Johnson target transformation wrapper
huber_gb = GradientBoostingRegressor(loss='huber', random_state=42)

robust_pipeline = Pipeline(steps=[
    ('engineering', HouseFeatureEngineer()),
    ('preprocessor', local_preprocessor),
    ('model', TransformedTargetRegressor(
        regressor=huber_gb,
        transformer=PowerTransformer(method='yeo-johnson')
    ))
])

y_cv_preds = cross_val_predict(robust_pipeline, X_dev_local, y_dev_local, cv=cv_local, n_jobs=-1)
residuals = y_dev_local - y_cv_preds

mae = mean_absolute_error(y_dev_local, y_cv_preds)
rmse = np.sqrt(mean_squared_error(y_dev_local, y_cv_preds))
print(f"Robust Yeo-Johnson Pipeline CV MAE: ${mae:,.2f}")
print(f"Robust Yeo-Johnson Pipeline CV RMSE: ${rmse:,.2f}")


Robust Yeo-Johnson Pipeline CV MAE: $15,771.05
Robust Yeo-Johnson Pipeline CV RMSE: $27,741.98
